# TDGBench — test des données locales

Ce notebook vérifie le premier jalon de la bibliothèque : catalogue local, manifestes, Gold Standards, splits, graphes et un entraînement GNN court sur Biomed.

## 1. Initialisation

Depuis la racine du dépôt, l'environnement notebook peut être installé avec :

```powershell
pip install -e ".[notebook]"
```

La cellule suivante fonctionne aussi sans installation editable en ajoutant `src/` au chemin Python.

In [ ]:
from __future__ import annotations

import json
import sys
from collections import Counter
from pathlib import Path

import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("Impossible de trouver la racine du projet tdg-bench")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_ROOT = PROJECT_ROOT / "src"
DATA_ROOT = PROJECT_ROOT / "benchmark_data"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from tdg_bench import TDGBench

bench = TDGBench(DATA_ROOT)
print("Projet :", PROJECT_ROOT)
print("Données :", DATA_ROOT)

## 2. Datasets et graphes disponibles

In [ ]:
datasets = bench.available_datasets()
assert datasets == ("biomed", "dbpedia")

for dataset in datasets:
    print(f"{dataset}: {bench.available_graphs(dataset)}")

assert bench.available_graphs("biomed") == ("GT2KG_tdg", "kggen")
assert bench.available_graphs("dbpedia") == ("GT2KG_tdg", "cg")

## 3. Résumé des manifestes

Le manifeste est le contrat qui sera réutilisé plus tard sur Hugging Face.

In [ ]:
manifest_rows = []
for dataset in datasets:
    manifest = bench._store.manifest(dataset)
    manifest_rows.append(
        {
            "dataset": dataset,
            "version": manifest.version,
            "graphs": ", ".join(sorted(manifest.graphs)),
            "split_count": len(manifest.splits),
            "split_seeds": sorted(manifest.splits),
        }
    )

pd.DataFrame(manifest_rows)

## 4. Gold Standards

In [ ]:
gold_standards = {}
for dataset in datasets:
    manifest = bench._store.manifest(dataset)
    table = pd.read_excel(manifest.gold_standard.path)
    required = {"term", "label"}
    assert required.issubset(table.columns), (dataset, table.columns.tolist())
    assert not table[["term", "label"]].isna().any().any()
    gold_standards[dataset] = table
    print(
        f"{dataset}: {len(table):,} termes, "
        f"{table['label'].nunique()} classes"
    )
    display(table.head(3))
    display(table["label"].value_counts().sort_index().rename("count").to_frame())

## 5. Validation des dix splits

Chaque split doit couvrir tout le Gold Standard, sans doublon ni chevauchement entre train, validation et test.

In [ ]:
split_rows = []
for dataset in datasets:
    manifest = bench._store.manifest(dataset)
    expected = set(range(len(gold_standards[dataset])))
    assert len(manifest.splits) == 10

    for seed, split_asset in sorted(manifest.splits.items()):
        split = json.loads(split_asset.path.read_text(encoding="utf-8"))
        train = set(split["train_idx"])
        val = set(split["val_idx"])
        test = set(split["test_idx"])
        assert len(train) == len(split["train_idx"])
        assert len(val) == len(split["val_idx"])
        assert len(test) == len(split["test_idx"])
        assert not (train & val or train & test or val & test)
        assert train | val | test == expected
        split_rows.append(
            {
                "dataset": dataset,
                "seed": seed,
                "train": len(train),
                "validation": len(val),
                "test": len(test),
            }
        )

split_summary = pd.DataFrame(split_rows)
display(split_summary)
print("Tous les splits sont valides.")

## 6. Graphes et lecture progressive

La fonction ci-dessous lit seulement les premiers objets d'un tableau JSON. Cela évite de charger le TDG DBpedia de 624 Mo entièrement en RAM.

In [ ]:
def first_json_array_records(path: Path, limit: int = 3, chunk_size: int = 64 * 1024):
    decoder = json.JSONDecoder()
    buffer = ""
    position = 0
    started = False
    with path.open(encoding="utf-8") as handle:
        while len(buffer) < chunk_size:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            buffer += chunk
            if "[" in buffer:
                position = buffer.index("[") + 1
                started = True
                break
        if not started:
            raise ValueError(f"{path} n'est pas un tableau JSON")

        records = []
        while len(records) < limit:
            while True:
                while position < len(buffer) and buffer[position] in " \r\n\t,":
                    position += 1
                try:
                    value, end = decoder.raw_decode(buffer, position)
                    records.append(value)
                    position = end
                    break
                except json.JSONDecodeError:
                    chunk = handle.read(chunk_size)
                    if not chunk:
                        raise
                    buffer = buffer[position:] + chunk
                    position = 0
        return records


graph_rows = []
for dataset in datasets:
    for graph_name in bench.available_graphs(dataset):
        graph = bench.get_graph(dataset, graph_name, verify=False)
        sample = first_json_array_records(graph.path)
        for index, record in enumerate(sample):
            missing = {"subject", "predicate", "object"} - set(record)
            assert not missing, (dataset, graph_name, index, missing)
        graph_rows.append(
            {
                "dataset": dataset,
                "graph": graph_name,
                "size_mb": round(graph.path.stat().st_size / 1_000_000, 2),
                "sample_predicates": dict(Counter(row["predicate"] for row in sample)),
                "path": str(graph.path),
            }
        )

pd.DataFrame(graph_rows)

## Test réel de `get_data()`

Cette cellule prépare le graphe biomédical GT2KG comme un objet PyTorch Geometric, applique le split 42 et utilise des features aléatoires déterministes de dimension 32.

In [ ]:
bundle = bench.get_data(
    dataset="biomed",
    graph="GT2KG_tdg",
    split_seed=42,
    feature_dim=32,
    random_seed=42,
)

assert bundle.coverage == 1.0
assert bundle.split_sizes == {"train": 103, "validation": 103, "test": 826}
assert bundle.data.x.shape[1] == 32
assert bundle.data.num_classes == 8
assert bundle.data.num_relations == len(bundle.index_to_relation)

print(bundle.data)
print("Couverture Gold Standard :", f"{bundle.coverage:.1%}")
print("Tailles du split :", bundle.split_sizes)
print("Classes :", bundle.label_names)

## 7. Test réel de l'évaluation GNN

Cette exécution courte valide tout le chemin `get_data → entraînement → early stopping → métriques`. Les valeurs ne constituent pas encore un résultat scientifique : seulement cinq epochs et des features aléatoires sont utilisés.

In [ ]:
print("Modèles intégrés :", bench.available_models())

result = bench.evaluate(
    dataset="biomed",
    graph="GT2KG_tdg",
    model="GCN",
    split_seed=42,
    feature_dim=32,
    hidden_dim=16,
    max_epochs=5,
    patience=3,
    random_seed=42,
    device="auto",
)

display(pd.DataFrame({
    "train": result.train.__dict__,
    "validation": result.validation.__dict__,
    "test": result.test.__dict__,
}).T)
print(f"Meilleur epoch : {result.best_epoch}; device : {result.device}")

## 8. Vérification SHA-256 optionnelle

Cette opération relit les quatre graphes complets. Elle est désactivée par défaut pour garder le notebook rapide.

In [ ]:
VERIFY_ALL_CHECKSUMS = False

if VERIFY_ALL_CHECKSUMS:
    for dataset in datasets:
        for graph_name in bench.available_graphs(dataset):
            graph = bench.get_graph(dataset, graph_name, verify=True)
            print(f"OK: {dataset}/{graph_name} — {graph.asset.sha256}")
else:
    print("Vérification complète ignorée. Mettre VERIFY_ALL_CHECKSUMS=True pour l'activer.")

## Résultat attendu

Si toutes les cellules passent :

- les deux datasets sont découverts ;
- les quatre graphes sont résolus ;
- les Gold Standards ont le bon schéma ;
- les vingt splits sont complets et disjoints ;
- les graphes commencent par des enregistrements `subject/predicate/object` ;
- `get_data()` construit le graphe PyG et applique le split pré-généré ;
- une évaluation GCN courte s'exécute de bout en bout.

Le jalon suivant sera la distribution des mêmes artefacts via Hugging Face.